# Scenario 3 Workbench: Multi-Agent Research System

Welcome to the hands-on offline lab for **Scenario 3**. Read [`SCENARIO_GUIDE.md`](SCENARIO_GUIDE.md) first for the Hub-and-Spoke architecture diagram, core rules, and 6 practice questions.

### What You Will Observe in This Lab
This notebook runs [`simulation.py`](simulation.py) offline so you can see how `ResearchCoordinator`:
1. Preserves conflicting statistics (`$1.2B` vs. `$1.4B`) side by side with sources instead of averaging them.
2. Treats zero search hits (`VALID_EMPTY`) as complete coverage, while flagging timeouts (`UPSTREAM_FAILURE`) as `PARTIAL_COVERAGE`.
3. Detects missing topics in the coordinator's task breakdown and merges claims without losing attribution.

**Memory Hook:** *"Split wide, pass context, fail loudly, cite everything."*

In [ ]:
# Step 1: Environment Setup & Simulation Import
import sys
from pathlib import Path

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scenarios.scenario_3_multi_agent_research.simulation import *

print(' Scenario Workbench Initialized: Scenario 3: Multi-Agent Research & Synthesis System')
print(' Zero live API calls, 100% offline determinism')


### Step 2: Preserve Conflicting Figures & Handle Valid Empty Searches
Run a parallel investigation where the SEC and Market subagents report different revenue numbers and the Patent subagent finds zero matches. Confirm that coverage remains `COMPLETE` and both revenue figures are preserved.

In [ ]:
# Test Consensus & Conflict Preservation
coord = ResearchCoordinator()

report = coord.run_parallel_investigation(
    "Acme Corp Earnings",
    {"sec_agent": "success", "market_agent": "success", "patent_agent": "empty"}
)
print("Coverage Status:", report.coverage_status)
print("Conflicts Preserved:", report.conflicts_preserved)
assert report.coverage_status == "COMPLETE"
assert len(report.conflicts_preserved) == 1
assert "$1.2B" in report.conflicts_preserved[0]["guidance"]
assert "$1.4B" in report.conflicts_preserved[0]["guidance"]



### Step 3: Handle Upstream Timeouts Without Hiding Gaps
Simulate a `504` timeout on the Patent subagent and confirm that the coordinator marks the report `PARTIAL_COVERAGE` and explicitly lists the omitted source.

In [ ]:
# Test Upstream Timeout Error Handling
coord_err = ResearchCoordinator()
report_err = coord_err.run_parallel_investigation(
    "Quantum Security",
    {"sec_agent": "success", "market_agent": "success", "patent_agent": "timeout"}
)
print("Partial Coverage Status:", report_err.coverage_status)
print("Source Omissions:", report_err.source_omissions)
assert report_err.coverage_status == "PARTIAL_COVERAGE"
assert report_err.confidence_score < 1.0
assert any("patent_agent" in o for o in report_err.source_omissions)



### Step 4: Test Coverage Checks, Structured Errors & Claim Merging
Verify that `check_decomposition_coverage` catches missed topics (like `music`), failed tools return retry metadata, and `merge_claims` flags conflicting numbers while preserving every source URL and date.

In [ ]:
check = ResearchCoordinator.check_decomposition_coverage(
    ["visual arts", "music"], ["survey visual arts trends"]
)
assert check["missing_topics"] == ["music"]
failed = SpecializedSubagent("patent_agent", "PATENTS").execute_isolated_task("q", "timeout")
assert failed.structured_error["is_retryable"] is True
claims = [
    {"claim": "revenue", "value": "$1.2B", "source_name": "10-Q",
     "source_url": "u1", "date": "2024-10-01"},
    {"claim": "revenue", "value": "$1.4B", "source_name": "Release",
     "source_url": "u2", "date": "2024-10-15"},
]
assert merge_claims(claims)[0]["status"] == "CONFLICT"
print("Coverage gap:", check["missing_topics"])


### Lab Complete
All offline checks passed. For full unit tests, run `uv run pytest scenarios/scenario_3_multi_agent_research/test_scenario_3_simulation.py` in WSL Ubuntu-26.04.